# Evaluation — Epic on FHIR Model

Deployment job **evaluation task**. Loads a registered model version, runs
traced predictions against the Epic FHIR sandbox (GET and POST payloads),
validates JSON serialization, and logs metrics to the Unity Catalog model
version page.

**Required job parameters**: `model_name`, `model_version`

This task **fails** (raises an exception) if any validation check does not
pass, which blocks the downstream approval and deployment tasks.

In [0]:
dbutils.widgets.text("model_name", "", "Model Name (catalog.schema.model)")
dbutils.widgets.text("model_version", "", "Model Version")

model_name = dbutils.widgets.get("model_name")
model_version = dbutils.widgets.get("model_version")

assert model_name, "model_name parameter is required"
assert model_version, "model_version parameter is required"

print(f"Evaluating {model_name} v{model_version}")

In [0]:
import json
import os
import sys
import random
from datetime import date, timedelta

import pandas as pd
import mlflow

# Add src directory to path for smart_on_fhir imports (needed for model loading)
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
if notebook_dir not in sys.path:
    sys.path.append(notebook_dir)

In [0]:
# The model's load_context reads secrets from env vars (same as model serving).
# These must be set before loading the model.
try:
    os.environ["EPIC_CLIENT_ID"] = dbutils.secrets.get("epic_on_fhir_oauth_keys", "client_id")
    os.environ["EPIC_PRIVATE_KEY"] = dbutils.secrets.get("epic_on_fhir_oauth_keys", "private_key")
    os.environ["EPIC_KID"] = dbutils.secrets.get("epic_on_fhir_oauth_keys", "kid")
    print("✓ Secrets loaded from scope")
except Exception as e:
    print(f"⚠ Could not load secrets: {e}")
    print("  Ensure epic_on_fhir_oauth_keys scope is accessible")

In [0]:
# MLflow 3: set_active_model links all subsequent traces to this model version.
# Traces and metrics will be visible on the UC model version page.
mlflow.set_active_model(name=model_name)

In [0]:
model_uri = f"models:/{model_name}/{model_version}"
print(f"Loading model from {model_uri}")
logged_model = mlflow.pyfunc.load_model(model_uri)
print(f"✓ Model loaded successfully")

In [0]:
def generate_test_payloads():
    """Generate FHIR test payloads: 1 GET (Patient) + 2 POSTs (Observation, AllergyIntolerance)."""
    _obs_time = f"{random.randint(0, 23):02d}:{random.randint(0, 59):02d}:{random.randint(0, 59):02d}"
    _recorded_date = (date(2024, 1, 1) + timedelta(days=random.randint(0, 365))).isoformat()

    observation_payload = {
        "resourceType": "Observation", "status": "final",
        "category": [{"coding": [{"system": "http://hl7.org/fhir/observation-category", "code": "vital-signs", "display": "Vital Signs"}], "text": "Vital Signs"}],
        "code": {"coding": [{"system": "urn:oid:1.2.840.114350.1.13.0.1.7.2.707679", "code": "8", "display": "Heart Rate"}], "text": "Heart Rate"},
        "subject": {"reference": "Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
        "encounter": {"reference": "Encounter/e0u1fd.jUCNqz8ZQuTaMtsQ3"},
        "effectiveDateTime": f"2019-09-05T{_obs_time}Z",
        "valueQuantity": {"value": 75},
    }
    allergy_payload = {
        "resourceType": "AllergyIntolerance",
        "clinicalStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/allergyintolerance-clinical", "code": "active", "display": "Active"}], "text": "Active"},
        "verificationStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/allergyintolerance-verification", "code": "unconfirmed", "display": "Unconfirmed"}], "text": "Unconfirmed"},
        "type": "allergy", "category": ["medication"], "criticality": "low",
        "code": {"coding": [{"system": "http://www.nlm.nih.gov/research/umls/rxnorm", "code": "7980", "display": "Penicillin G"}], "text": "Penicillin"},
        "patient": {"reference": "Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
        "recorder": {"reference": "Practitioner/eM5CWtq15N0WJeuCet5bJlQ3", "display": "Physician Family Medicine, MD"},
        "recordedDate": _recorded_date,
        "reaction": [{"manifestation": [{"coding": [{"system": "http://snomed.info/sct", "code": "247472004", "display": "Hives"}], "text": "Hives"}]}],
    }
    return pd.DataFrame([
        {"http_method": "get", "resource": "Patient", "action": "T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
        {"http_method": "post", "resource": "Observation", "action": "", "data": json.dumps(observation_payload)},
        {"http_method": "post", "resource": "AllergyIntolerance", "action": "", "data": json.dumps(allergy_payload)},
    ])


@mlflow.trace
def traced_predict(model, df: pd.DataFrame):
    """Run prediction with MLflow tracing linked to the active model."""
    return model.predict(df)

In [0]:
payloads = generate_test_payloads()
results = {}

# Run each payload individually to capture per-request metrics
for idx, row in payloads.iterrows():
    resource = row["resource"]
    method = row["http_method"]
    label = f"{method}_{resource}".lower()

    input_df = pd.DataFrame([row.where(row.notna(), None)])
    result = traced_predict(logged_model, input_df)

    if isinstance(result, list) and len(result) > 0:
        resp = result[0] if isinstance(result[0], dict) else {"response": result[0]}
    else:
        resp = {"response": str(result)}

    results[label] = resp
    status = resp.get("response_status_code", -1)
    duration = resp.get("response_time_seconds", -1)
    print(f"{method.upper()} {resource}: status={status}, duration={duration:.3f}s")

In [0]:
# Validate that all responses are JSON-serializable (required for model serving)
json_valid = True
for label, resp in results.items():
    try:
        json.dumps(resp)
    except (TypeError, ValueError) as e:
        print(f"✗ JSON serialization failed for {label}: {e}")
        json_valid = False

if json_valid:
    print("✓ All responses are JSON-serializable")

In [0]:
# Log validation metrics. These will be visible on the UC model version page
# via the MLflow 3 deployment job integration.
metrics = {}

for label, resp in results.items():
    metrics[f"validation.{label}.status_code"] = resp.get("response_status_code", -1)
    metrics[f"validation.{label}.response_time_seconds"] = resp.get("response_time_seconds", -1)

metrics["validation.json_serializable"] = 1.0 if json_valid else 0.0

# Check expected status codes: GET Patient → 200, POST Observation → 201, POST AllergyIntolerance → 201
_get_ok = results.get("get_patient", {}).get("response_status_code", -1) == 200
_post_obs_ok = results.get("post_observation", {}).get("response_status_code", -1) == 201
# AllergyIntolerance POST returns 201 but read-back may fail (known Epic sandbox issue)
_post_allergy_ok = results.get("post_allergyintolerance", {}).get("response_status_code", -1) == 201

metrics["validation.get_patient_pass"] = 1.0 if _get_ok else 0.0
metrics["validation.post_observation_pass"] = 1.0 if _post_obs_ok else 0.0
metrics["validation.post_allergyintolerance_pass"] = 1.0 if _post_allergy_ok else 0.0
metrics["validation.all_passed"] = 1.0 if (_get_ok and _post_obs_ok and _post_allergy_ok and json_valid) else 0.0

with mlflow.start_run():
    mlflow.log_metrics(metrics)
    print(f"✓ Logged {len(metrics)} metrics to model version")

for k, v in sorted(metrics.items()):
    print(f"  {k} = {v}")

In [0]:
# Fail the task if critical validations did not pass.
# This blocks the approval and deployment tasks in the deployment job.
assert _get_ok, f"GET Patient failed: status={results.get('get_patient', {}).get('response_status_code', 'N/A')}"
assert _post_obs_ok, f"POST Observation failed: status={results.get('post_observation', {}).get('response_status_code', 'N/A')}"
assert _post_allergy_ok, f"POST AllergyIntolerance failed: status={results.get('post_allergyintolerance', {}).get('response_status_code', 'N/A')}"
assert json_valid, "JSON serialization validation failed"

print(f"✓ All validation checks passed for {model_name} v{model_version}")
dbutils.notebook.exit(json.dumps({"model_name": model_name, "model_version": model_version, "validation": "passed"}))